# Large-Scale Empirical Audit: Passage-to-Proposition Logic Error Mapping (Chen et al., EMNLP 2024)
**Dự án:** LP-Prop — Logic-Preserving Proposition Retrieval  
**Mục tiêu cốt lõi:**
Quét luồng (Stream) trực tiếp song song cả 2 kho ngữ liệu của tác giả trên Hugging Face:
1. `chentong00/factoid-wiki-passage`: Kho chứa các đoạn văn (Passages).
2. `chentong00/factoid-wiki`: Kho chứa các mệnh đề nguyên tử tương ứng do mô hình Propositionizer sinh ra (Propositions).

**Thuật toán ánh xạ 1-1 (Aligned Shard Mapping):**
- Với mỗi shard $i$:
  1. Quét `passage` shard $i$ để lọc các đoạn văn chứa liên từ logic (`unless`, `provided that`, `except that`, `only if`, `as long as`).
  2. Quét đồng thời `proposition` shard $i$ tương ứng để gom toàn bộ propositions theo đúng mã `pid` (Passage ID).
  3. Kiểm định trực tiếp: Vế điều kiện / ngoại lệ có bị mô hình Propositionizer của tác giả **cắt bỏ hoàn toàn** hay không (`keyword_preserved == False`).
  4. Lượng hóa chính xác tỷ lệ lỗi logic thực tế trên quy mô hàng triệu đoạn văn.


## 0. Cài đặt thư viện & Cấu hình môi trường (Setup & Environment)


In [ ]:
# Cài đặt huggingface_hub nếu chưa có
# !pip install -q huggingface_hub orjson pandas matplotlib seaborn tqdm

import os
import gc
import re
import json
import time
import math
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from huggingface_hub import HfFileSystem

print("[+] Môi trường sẵn sàng cho Large-Scale Passage-to-Proposition Mapping Audit!")


## 1. Cấu hình chế độ quét (Audit Scale Configuration)
Chọn số lượng shards muốn duyệt đối soát:
- **`pilot` (10 shards)**: ~200k passages $\to$ Chạy thử nhanh mất ~2-3 phút.
- **`sample` (30 shards)**: ~600k passages $\to$ Chạy mất ~5-8 phút (Rất đẹp trên Kaggle).
- **`large` (100 shards)**: ~2 triệu passages $\to$ Chạy mất ~15-20 phút.
- **`full` (1000 shards)**: ~21 triệu passages $\to$ Quét toàn diện 100% Wikipedia.


In [ ]:
# Chọn chế độ quét: "pilot", "sample", "large", hoặc "full"
AUDIT_MODE = "sample"  

MODE_SHARDS = {
    "pilot": 10,
    "sample": 30,
    "large": 100,
    "full": 1000
}

NUM_SHARDS_TO_SCAN = MODE_SHARDS.get(AUDIT_MODE, 30)
MAX_WORKERS = 4  # Số luồng song song (đảm bảo không bị giới hạn rate-limit)
OUT_DIR = "/kaggle/working/factoidwiki_audit_results" if os.path.exists("/kaggle/working") else "factoidwiki_audit_results"
os.makedirs(OUT_DIR, exist_ok=True)

# Repository của tác giả trên Hugging Face
REPO_PASSAGE = "chentong00/factoid-wiki-passage"
REPO_PROP = "chentong00/factoid-wiki"

print(f"[*] Chế độ quét: {AUDIT_MODE.upper()} | Số lượng shards: {NUM_SHARDS_TO_SCAN} / 1000 shards")
print(f"[*] Shard Passage Repo: {REPO_PASSAGE}")
print(f"[*] Shard Proposition Repo: {REPO_PROP}")
print(f"[*] Thư mục lưu kết quả: {OUT_DIR}")


## 2. Định nghĩa Phổ Liên từ Logic & Ngoại lệ (Logic Triggers)


In [ ]:
LOGIC_PATTERNS = [
    (r"\bunless\b", "unless"),
    (r"\bexcept (that|when|if|in)\b", "except"),
    (r"\bprovided that\b", "provided that"),
    (r"\bproviding that\b", "providing that"),
    (r"\bonly if\b", "only if"),
    (r"\bas long as\b", "as long as"),
    (r"\bso long as\b", "so long as"),
    (r"\bon condition that\b", "on condition that")
]

COMBINED_REGEX = re.compile("|".join([pat for pat, _ in LOGIC_PATTERNS]), re.IGNORECASE)

def detect_logic_trigger(text):
    m = COMBINED_REGEX.search(text)
    if m:
        return m.group(0).lower()
    return None

print(f"[+] Đã cấu hình {len(LOGIC_PATTERNS)} họ liên từ logic.")


## 3. Thuật toán Quét & Ánh xạ Cặp Shard (Aligned Shard Scanner)
Hàm `scan_aligned_shard_pair` thực hiện:
1. Đọc stream shard `passage` $\to$ tìm các đoạn văn có liên từ logic $\to$ ghi nhớ `pid`.
2. Đọc stream shard `proposition` tương ứng $\to$ gom các câu proposition cùng `pid`.
3. Kiểm tra xem liên từ logic có xuất hiện trong bất kỳ câu proposition nào không.


In [ ]:
fs = HfFileSystem()

print("[*] Đang kết nối Hugging Face để lấy danh sách shard...")
all_p_files = fs.ls(f"datasets/{REPO_PASSAGE}/data", detail=True)
shard_files = sorted([f["name"].split("/data/")[-1] for f in all_p_files if f["name"].endswith(".jsonl")])

target_shards = shard_files[:NUM_SHARDS_TO_SCAN]
print(f"[+] Chuẩn bị duyệt qua {len(target_shards)} cặp shard song song...")

def parse_prop_line(d):
    """Trích xuất text của proposition và PID chuẩn xác"""
    uid = d.get("id", "")
    pid = uid[:25]  # Cắt 25 ký tự đầu lấy Passage ID
    c = d.get("contents", "")
    m = d.get("metadata", {})
    cs, ce = m.get("content_span", [None, None])
    if cs is not None and ce is not None:
        p_text = c[cs:ce].strip()
    elif "\n" in c:
        p_text = c.split("\n", 1)[1].strip()
    else:
        p_text = c.strip()
    return pid, p_text

def scan_aligned_shard_pair(shard_fname):
    """
    Xử lý 1 cặp shard (Passage vs. Proposition) tương ứng 1-1.
    """
    matched_passages = {}
    passages_count = 0
    props_count = 0
    results_in_shard = []
    
    pass_url = f"datasets/{REPO_PASSAGE}/data/{shard_fname}"
    prop_url = f"datasets/{REPO_PROP}/data/{shard_fname}"
    
    # 1. Quét Passage Shard
    try:
        with fs.open(pass_url, "rt", encoding="utf-8") as f:
            for line in f:
                passages_count += 1
                try:
                    d = json.loads(line)
                    text = d.get("contents", "")
                    trig = detect_logic_trigger(text)
                    if trig:
                        uid = d.get("id", "")
                        pid = uid[:25]
                        matched_passages[pid] = {
                            "pid": pid,
                            "title": d.get("title", ""),
                            "passage_text": text,
                            "trigger": trig
                        }
                except:
                    continue
    except Exception as e:
        print(f"[!] Lỗi đọc passage shard {shard_fname}: {e}")
        return [], passages_count, props_count

    if not matched_passages:
        return [], passages_count, props_count

    # 2. Quét Proposition Shard tương ứng để gom propositions theo PID (25 ký tự đầu của id)
    props_by_pid = defaultdict(list)
    try:
        with fs.open(prop_url, "rt", encoding="utf-8") as f:
            for line in f:
                props_count += 1
                try:
                    d = json.loads(line)
                    pid, p_text = parse_prop_line(d)
                    if pid in matched_passages:
                        props_by_pid[pid].append(p_text)
                except:
                    continue
    except Exception as e:
        print(f"[!] Lỗi đọc proposition shard {shard_fname}: {e}")

    # 3. Đối chiếu kiểm tra sự rơi rụng liên từ logic
    for pid, p_info in matched_passages.items():
        props = props_by_pid.get(pid, [])
        trig = p_info["trigger"]
        
        # Kiểm tra xem từ khóa logic có tồn tại trong bất kỳ câu proposition nào không
        preserved = any(re.search(r"\b" + re.escape(trig) + r"\b", p, re.IGNORECASE) for p in props)
        
        results_in_shard.append({
            "pid": pid,
            "title": p_info["title"],
            "keyword": trig,
            "keyword_preserved_in_props": preserved,
            "passage_text": p_info["passage_text"],
            "propositions": "\n".join([f"[{i+1}] {p}" for i, p in enumerate(props)]),
            "num_propositions": len(props),
            "shard": shard_fname
        })
        
    return results_in_shard, passages_count, props_count

print("[+] Khởi tạo thành công hàm Aligned Shard Scanner (Đã fix lỗi ánh xạ PID 25 ký tự)!")


## 4. Chạy Quét Đối Soát Toàn Diện (Execute Aligned Audit)


In [ ]:
print(f"[*] Bắt đầu quét đối soát {len(target_shards)} cặp shard...")
start_time = time.time()

all_audit_results = []
total_passages_scanned = 0
total_props_scanned = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(scan_aligned_shard_pair, fname): fname for fname in target_shards}
    pbar = tqdm(as_completed(futures), total=len(target_shards), desc="Auditing Shard Pairs")
    
    for future in pbar:
        res, p_cnt, pr_cnt = future.result()
        all_audit_results.extend(res)
        total_passages_scanned += p_cnt
        total_props_scanned += pr_cnt
        
        dropped_count = sum(1 for r in all_audit_results if not r["keyword_preserved_in_props"])
        pbar.set_postfix({
            "Passages": f"{total_passages_scanned:,}",
            "Logic Cases": len(all_audit_results),
            "Dropped": dropped_count
        })

elapsed_min = (time.time() - start_time) / 60
print(f"\n[+] HOÀN TẤT QUÉT ĐỐI SOÁT TRONG {elapsed_min:.2f} PHÚT!")
print(f"    -> Tổng số Passage quét qua: {total_passages_scanned:,}")
print(f"    -> Tổng số Proposition quét qua: {total_props_scanned:,}")
print(f"    -> Số câu chứa cấu trúc logic: {len(all_audit_results):,}")


## 5. Lượng hóa Tỷ lệ Cắt cụt Điều kiện Logic (Quantify Drop Rate)


In [ ]:
df_audit = pd.DataFrame(all_audit_results)
out_csv = os.path.join(OUT_DIR, f"factoidwiki_aligned_audit_{AUDIT_MODE}.csv")
df_audit.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"[+] Đã xuất toàn bộ kết quả đối chiếu ra file: {out_csv}")

total_logic = len(df_audit)
dropped = df_audit[df_audit["keyword_preserved_in_props"] == False]
dropped_count = len(dropped)
error_rate = (dropped_count / total_logic * 100) if total_logic > 0 else 0

print("="*75)
print("               BÁO CÁO TỶ LỆ LỖI LOGIC TRÊN TẬP DỮ LIỆU TÁC GIẢ")
print("="*75)
print(f"Tổng số đoạn văn chứa liên từ logic : {total_logic:,}")
print(f"Số đoạn văn BỊ CẮT BỎ HOÀN TOÀN ĐIỀU KIỆN : {dropped_count:,} ({error_rate:.1f}%)")
print("\nTop các liên từ bị mô hình Propositionizer của tác giả làm mất nhiều nhất:")
print(dropped["keyword"].value_counts().head(10).to_string())
print("="*75)


## 6. Trực quan hóa Biểu đồ Báo cáo Học thuật (Visual Charts)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.set_theme(style="whitegrid")

# 1. Biểu đồ tròn: Tỷ lệ mất điều kiện vs Giữ được
labels = ['Giữ được điều kiện', 'Bị cắt bỏ điều kiện (LỖI)']
sizes = [total_logic - dropped_count, dropped_count]
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors, explode=(0, 0.1), textprops={'fontsize': 11, 'weight': 'bold'})
axes[0].set_title(f'Tỷ lệ Bảo tồn Logic của Propositionizer\n(Duyệt {total_passages_scanned:,} Passages)', fontsize=13, weight='bold')

# 2. Biểu đồ cột: Các từ khóa bị cắt cụt nhiều nhất
top_dropped_kw = dropped["keyword"].value_counts().head(6)
sns.barplot(x=top_dropped_kw.values, y=top_dropped_kw.index, ax=axes[1], palette="Reds_r")
axes[1].set_title('Top các Liên từ Logic bị Cắt cụt Nhiều nhất', fontsize=13, weight='bold')
axes[1].set_xlabel('Số lượng ca lỗi (Cases)')

for i, v in enumerate(top_dropped_kw.values):
    axes[1].text(v + 1, i, f"{v:,}", va="center", fontsize=10, weight="bold")

plt.tight_layout()
chart_file = os.path.join(OUT_DIR, "figure_proposition_drop_rate.png")
plt.savefig(chart_file, dpi=300)
plt.show()
print(f"[+] Đã lưu biểu đồ học thuật vào: {chart_file}")


## 7. Tổng kết Báo cáo Khoa học (Academic Summary Deliverable)


In [ ]:
summary_metrics = {
    "Audit Scope": f"{AUDIT_MODE.upper()} ({NUM_SHARDS_TO_SCAN} shards)",
    "Total Passages Scanned": f"{total_passages_scanned:,}",
    "Total Propositions Scanned": f"{total_props_scanned:,}",
    "Total Logic Passages": f"{total_logic:,}",
    "Condition Truncation Errors": f"{dropped_count:,}",
    "Global Error Rate": f"{error_rate:.2f}%",
    "Estimated Errors across Full 21M Wikipedia": f"{int(21015324 * (total_logic/max(1, total_passages_scanned)) * (error_rate/100)):,} passages",
    "Top Vulnerable Keyword": top_dropped_kw.index[0] if len(top_dropped_kw)>0 else "N/A"
}

summary_json_path = os.path.join(OUT_DIR, "full_audit_summary.json")
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, indent=2, ensure_ascii=False)

print("[+] Đã hoàn tất đóng gói kết quả nghiên cứu quy mô lớn!")
print(json.dumps(summary_metrics, indent=2, ensure_ascii=False))
